# 01 - Exploratory Data Analysis (EDA)

This notebook explores the NSL-KDD / synthetic dataset used by the Adversarial Robust IDS.

**Sections:**
1. Load data
2. Dataset shape, columns, data types
3. Label distribution
4. Feature statistics and correlations
5. Missing values analysis

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing.data_loader import DataLoader, NSL_KDD_COLUMNS, CATEGORY_MAP
from src.utils.config import load_config

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

print('Imports loaded successfully.')

## 1. Load Data

We attempt to load the NSL-KDD dataset. If the raw files are not available,
we fall back to generating synthetic data that mirrors the NSL-KDD schema.

In [ ]:
config = load_config('../config/config.yaml')
loader = DataLoader()

try:
    df = loader.load(config)
    data_source = 'NSL-KDD'
except FileNotFoundError:
    print('NSL-KDD files not found. Generating synthetic data...')
    df = loader.generate_synthetic(n_samples=5000)
    data_source = 'Synthetic'

print(f'Data source: {data_source}')
print(f'Shape: {df.shape}')
df.head()

## 2. Dataset Shape, Columns, and Data Types

In [ ]:
print(f'Number of samples : {df.shape[0]}')
print(f'Number of features: {df.shape[1]}')
print()
print('--- Column data types ---')
print(df.dtypes.value_counts())
print()
print('Categorical columns:', df.select_dtypes(include='object').columns.tolist())
print('Numeric columns    :', len(df.select_dtypes(include=np.number).columns))

In [ ]:
df.info()

## 3. Label Distribution

The `attack_category` column maps raw labels to 5 classes:
Normal, DoS, Probe, R2L, U2R.

In [ ]:
label_counts = df['attack_category'].value_counts()
print('Label distribution:')
print(label_counts)
print()
print('Label percentages:')
print((label_counts / len(df) * 100).round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
colors = ['#2ecc71', '#e74c3c', '#3498db', '#f39c12', '#9b59b6']
label_counts.plot(kind='bar', ax=axes[0], color=colors[:len(label_counts)])
axes[0].set_title('Attack Category Distribution (Count)')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Pie chart
axes[1].pie(label_counts, labels=label_counts.index, autopct='%1.1f%%',
            colors=colors[:len(label_counts)], startangle=90)
axes[1].set_title('Attack Category Distribution (%)')

plt.tight_layout()
plt.show()

In [ ]:
# Binary label distribution: Normal vs Attack
binary_counts = pd.Series({
    'Normal': (df['attack_category'] == 'Normal').sum(),
    'Attack': (df['attack_category'] != 'Normal').sum()
})

fig, ax = plt.subplots(figsize=(6, 4))
binary_counts.plot(kind='bar', color=['#2ecc71', '#e74c3c'], ax=ax)
ax.set_title('Binary Label Distribution')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=0)
for i, v in enumerate(binary_counts):
    ax.text(i, v + 20, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Feature Statistics and Correlations

In [ ]:
numeric_df = df.select_dtypes(include=np.number)
print(f'Numeric features: {numeric_df.shape[1]}')
print()
numeric_df.describe().T.head(20)

In [ ]:
# Distribution of a few key features
key_features = ['duration', 'src_bytes', 'dst_bytes', 'count', 'srv_count']
available = [f for f in key_features if f in numeric_df.columns]

if available:
    fig, axes = plt.subplots(1, len(available), figsize=(4 * len(available), 4))
    if len(available) == 1:
        axes = [axes]
    for ax, feat in zip(axes, available):
        df[feat].hist(bins=50, ax=ax, color='steelblue', edgecolor='white')
        ax.set_title(feat)
        ax.set_xlabel('Value')
    plt.suptitle('Key Feature Distributions', y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# Correlation matrix (top numeric features)
top_numeric = numeric_df.iloc[:, :20]  # first 20 numeric features
corr = top_numeric.corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='coolwarm', center=0,
            linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix (First 20 Numeric Features)')
plt.tight_layout()
plt.show()

In [ ]:
# Highly correlated feature pairs (|r| > 0.8)
high_corr = []
for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):
        if abs(corr.iloc[i, j]) > 0.8:
            high_corr.append({
                'Feature 1': corr.columns[i],
                'Feature 2': corr.columns[j],
                'Correlation': round(corr.iloc[i, j], 4)
            })

if high_corr:
    print(f'Found {len(high_corr)} highly correlated pairs (|r| > 0.8):')
    pd.DataFrame(high_corr).sort_values('Correlation', ascending=False, key=abs)
else:
    print('No feature pairs with |r| > 0.8 found.')

## 5. Missing Values Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing Count', ascending=False)

print(f'Total missing values: {missing.sum()}')
print(f'Columns with missing values: {(missing > 0).sum()}')
print()

if missing.sum() > 0:
    print('Columns with missing data:')
    print(missing_summary[missing_summary['Missing Count'] > 0])
else:
    print('No missing values detected in the dataset.')

In [ ]:
# Check for infinities in numeric columns
inf_counts = np.isinf(numeric_df).sum()
total_inf = inf_counts.sum()
print(f'Total infinite values: {total_inf}')

if total_inf > 0:
    print('Columns with infinite values:')
    print(inf_counts[inf_counts > 0])

In [ ]:
# Duplicate rows
n_dupes = df.duplicated().sum()
print(f'Duplicate rows: {n_dupes} ({n_dupes / len(df) * 100:.2f}%)')

## Summary

Key observations from EDA:
- The dataset contains both numeric and categorical features matching the NSL-KDD schema.
- Class imbalance is present -- the preprocessing pipeline uses SMOTE to address this.
- Several features are highly correlated, motivating feature selection (mutual information).
- Missing and infinite values (if any) are handled during preprocessing.

Next: `02_preprocessing.ipynb` -- apply the full preprocessing pipeline.